In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import json
from sql_metadata import Parser

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("/home/yfwang/wyy/schema_routing/model_file/Qwen3-Embedding-4B", device=device)



In [ ]:
def load_corpus(file_path: str) -> list[str]:
    """
    读取包含JSON对象列表的文件，并将每个对象转换为描述性字符串。

    参数:
    - file_path (str): JSON文件的路径。

    返回:
    - list[str]: 由描述性字符串组成的语料库列表。
    """
    corpus = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # data 是一个列表
            
            # 遍历列表中的每一个JSON对象 (字典)
            for item in data:
                # 将键值对转换为 "key is value" 的形式
                parts = [f"{key} is {value}" for key, value in item.items()]
                
                # 用逗号和空格将它们连接成一个完整的字符串
                text_representation = ", ".join(parts)
                
                corpus.append(text_representation)
                
    except FileNotFoundError:
        print(f"错误: 文件 '{file_path}' 未找到。")
    except json.JSONDecodeError:
        print(f"错误: 文件 '{file_path}' 不是有效的JSON格式。")
        
    return corpus

In [ ]:
def load_queries(file_path: str) -> list[str], list[str]:
    questions = []
    answers = []
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        for item in data:
            question= item['question']
            query = item['query']
            answer = Parser(query).tables
            
            instruction = "Given a user's question, retrieve the most relevant table descriptions from the database."
            questions.append(f"Instruction: {instruction} \n Query: {question}")
            answers.append(answer)
    return questions, answers

In [ ]:
def find_most_similar(query: str, corpus_embeddings: torch.Tensor, model: SentenceTransformer, top_k: int = 5):
    """
    在一个给定的语料库向量中，为一个查询语句找到最相似的 top_k 个句子。

    参数:
    - query (str): 你要查询的句子。
    - corpus_embeddings (torch.Tensor): 已经提前计算好的语料库向量矩阵。
    - model (SentenceTransformer): 使用的句向量模型。
    - top_k (int): 返回最相似句子的数量。

    返回:
    - a list of tuples: 每个元组包含 (相似度分数, 句子在语料库中的索引)。
    """
    # 1. 将查询语句编码为向量
    query_embedding = model.encode(query, convert_to_tensor=True)

    # 2. 使用 util.cos_sim 计算查询向量与所有语料库向量的余弦相似度
    #    这个函数非常高效，利用了PyTorch的并行计算能力
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # 3. 使用 torch.topk 找到分数最高的 top_k 个结果的索引和分数
    #    这比手动排序然后切片更高效
    top_results = torch.topk(cos_scores, k=min(top_k, len(corpus_embeddings)))
    
    # top_results 是一个包含 (values, indices) 的元组
    return zip(top_results[0], top_results[1])

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import json
import os

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "/home/yfwang/wyy/schema_routing/model_file/Qwen3-Embedding-4B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# 使用 bfloat16 加载模型以节省显存，需要 Ampere 或更新的 GPU
# 如果你的 GPU 不支持 bfloat16，可以尝试 float16
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16)
model.to(device)
print(model)


In [7]:
import json
from sql_metadata import Parser


def load_queries(file_path: str):
    """原有的同步数据加载函数（备用）"""
    questions = []
    db_ids = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data:
                question= item['question']
                query = item['query']
                db_id = item['db_id']
                

                questions.append(question)
                db_ids.append(db_id)
    except FileNotFoundError:
        print(f"错误: 文件 '{file_path}' 未找到。")
    except json.JSONDecodeError:
        print(f"错误: 文件 '{file_path}' 不是有效的JSON格式。")
    return questions, db_ids

In [6]:
def load_final_queries(file_path: str):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        queries = data['final_query']
    return queries

In [10]:
import re

original_questions, original_dbs = load_queries("/home/yfwang/wyy/schema_routing/data/spider/spider_data/dev.json")
queries = load_final_queries("/home/yfwang/wyy/schema_routing/data/spider/spider_data/cache/dev_final_query_column_filtering.json")
if len(original_questions) != len(queries):
    print("original_questions and generated_queries have different lengths")
else:
    print("original_questions and generated_queries have the same length")

for i in range(len(original_questions)):
    if original_questions[i] != queries[i]['question']:
        print(f"original_questions[{i}] != generated_questions[{i}]")
        print(original_questions[i])
        print(queries[i]['question'])
        print("-"*100)
    if original_dbs[i] != queries[i]['answer']['db_id']:
        print(f"original_dbs[{i}] != generated_dbs[{i}]")
        print(original_dbs[i])
        print(queries[i]['answer']['db_id'])
        print("-"*100)
    cnt = len(queries[i]["original_schema"])
    if cnt != 5:
        print(f"original_dbs[{i}] != generated_dbs[{i}]")
        print(original_dbs[i])
        print(queries[i]['answer']['db_id'])
        print(cnt)
        print("-"*100)

print("done")

original_questions and generated_queries have the same length
done


In [14]:
def parse_prediction(pred_str: str):
    """
    解析预测字符串，提取 database 和 table。
    例如: "database: perpetrator, table: perpetrator" -> ("perpetrator", "perpetrator")
    """
    # 使用正则表达式来匹配 "database: value, table: value" 格式，忽略大小写和多余的空格
    match = re.search(r"database: (.*?), table: (.*?),", pred_str, re.IGNORECASE)
    if match:
        db_id = match.group(1).strip()
        table = match.group(2).strip()
        return db_id, table
    
    # 如果正则匹配失败，打印一个警告并返回 None
    print(f"警告: 无法解析预测字符串: '{pred_str}'")
    return None, None

In [ ]:
print(parse_prediction("database: music_2, table: Vocals, full_text: Database: music_2, Table: Vocals, Description: \"Band members' vocal roles in songs\" in music band management"))

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# --- 1. 定义路径 ---
# 基础模型的名称或路径
base_model_path = "/root/autodl-tmp/model_file/Qwen3-Embedding-4B"
# 你训练好的 LoRA 适配器路径
adapter_path = "/root/autodl-tmp/model_file/Qwen3-Embedding-4B-lora" 
# 你想将合并后的模型保存到的新路径
merged_model_path = "//root/autodl-tmp/model_file/Qwen3-Embedding-4B-lora-merged"

print("🚀 正在加载基础模型和 Tokenizer...")
# 加载基础模型和 tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
# 以 bfloat16 格式加载，可以节省内存
base_model = AutoModel.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto" # 自动将模型加载到可用设备（如GPU）
)

print("🚀 正在加载 LoRA 适配器...")
# 加载 LoRA 适配器
# 这会在基础模型之上加载 LoRA 权重，形成 PeftModel
model = PeftModel.from_pretrained(base_model, adapter_path)

print("🚀 正在合并 LoRA 权重...")
# 核心步骤：合并 LoRA 权重到基础模型中
# 这会返回一个标准的 transformers 模型，而不是 PeftModel
model = model.merge_and_unload()
print("✅ 合并完成！")


print(f"🚀 正在将合并后的完整模型保存到 {merged_model_path}...")
# 使用 save_pretrained 方法保存合并后的模型
# 这会保存完整的模型权重、配置文件等
model.save_pretrained(merged_model_path)
# 同时保存 tokenizer
tokenizer.save_pretrained(merged_model_path)

print(f"🎉 成功！合并后的模型已保存至 {merged_model_path}。你现在可以独立使用这个目录了。")

/root/miniconda3/envs/schlink/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 正在加载基础模型和 Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.76it/s]


🚀 正在加载 LoRA 适配器...
🚀 正在合并 LoRA 权重...
✅ 合并完成！
🚀 正在将合并后的完整模型保存到 //root/autodl-tmp/model_file/Qwen3-Embedding-4B-lora-merged...
🎉 成功！合并后的模型已保存至 //root/autodl-tmp/model_file/Qwen3-Embedding-4B-lora-merged。你现在可以独立使用这个目录了。


python data/spider/test-suite-sql-eval/evaluation.py --gold /home/yfwang/wyy/schema_routing/data/spider/spider_data/dev_gold.sql --pred /home/yfwang/wyy/schema_routing/data/spider/spider_data/cache/dev_final_query_column_filtering_sql.sql --etype all --db /home/yfwang/wyy/schema_routing/data/spider/spider_data/database --table /home/yfwang/wyy/schema_routing/data/spider/spider_data/tables.json --plug_value --keep_distinct

In [11]:
# 定义输入和输出文件名
input_filename = "/home/yfwang/wyy/schema_routing/data/spider/spider_data/cache/dev_final_query_column_filtering_sql.sql"
output_filename = "/home/yfwang/wyy/schema_routing/data/spider/spider_data/cache/dev_sql_final.sql"

try:
    # 使用 with 语句可以确保文件被正确关闭
    with open(input_filename, 'r', encoding='utf-8') as f_in, \
         open(output_filename, 'w', encoding='utf-8') as f_out:
        
        # 逐行读取输入文件
        for line in f_in:
            # 1. 使用制表符分割字符串，最多分割1次
            # 2. 取分割后的第一部分 [0]
            # 3. strip() 用于去除可能存在的前后空白
            sql_part = line.split('\t', 1)[0].strip()
            
            # 如果处理后sql部分不为空，则写入新文件
            if sql_part:
                f_out.write(sql_part + '\n')

    print(f"处理完成！结果已保存到 {output_filename}")

except FileNotFoundError:
    print(f"错误: 输入文件 '{input_filename}' 未找到。")
except Exception as e:
    print(f"处理过程中发生错误: {e}")

处理完成！结果已保存到 /home/yfwang/wyy/schema_routing/data/spider/spider_data/cache/dev_sql_final.sql
